# Segger cell size parameters (2D projection)

This notebook explains how to choose Segger's cell-size-related parameters in micron space (2D projection):
- `prediction_scale_factor` (linear polygon scaling)
- `area_low`, `area_high` (cell area filters for export)

Defaults in this repo:
- `prediction_scale_factor = 2.0`
- `area_low = 10` (um^2)
- `area_high = 1000` (um^2)


## What the scale factor means (linear, not area)

Segger scales each boundary polygon about its centroid by a linear factor `s`.
- Diameters/radii scale by `s`.
- Areas scale by `s^2`.

If you have areas:
`s = sqrt(A_cell / A_nucleus)`

If you have diameters:
`s = D_cell / D_nucleus`

This is why the scale factor should be chosen from *linear* sizes, not area ratios.


## Biology anchors (2D size context)

These references provide typical mammalian cell and nucleus size ranges:
- Typical animal cell diameter: ~10-20 um. Source: https://www.ncbi.nlm.nih.gov/books/NBK26880/
- Typical mammalian nucleus diameter: ~5-10 um. Source: https://bionumbers.hms.harvard.edu/bionumber.aspx?id=109953
- Nucleus occupies ~10% of total cell volume (mammals). Source: https://bionumbers.hms.harvard.edu/bionumber.aspx?id=113848
- Small cells (lymphocytes) are about ~7 um diameter. Source: https://www.ncbi.nlm.nih.gov/books/NBK535433/
- Larger cells (hepatocytes) are often ~16-27 um diameter. Source: https://pmc.ncbi.nlm.nih.gov/articles/PMC1866923/

Note: these are 3D size references. For 2D projections, area ratios will differ, but the *linear* scale factor remains the relevant quantity for Segger.


## Recommended scale-factor ranges

Rule of thumb (nucleus -> cell):
- Mixed epithelial tissues: s ~ 2.0 to 2.2
- Lymphoid-rich (nucleus dominates): s ~ 1.2 to 1.6
- Large-cell tissues (e.g., hepatocyte-rich): s ~ 2.5 to 3.5

The default `s=2.0` is a conservative middle-ground that expands nucleus-based polygons to a typical cell size without being overly aggressive.

If you already predict on *cell* boundaries (prediction_mode='cell'), use a smaller slack factor (e.g., 1.05-1.25).


## Area filters (area_low / area_high) in um^2

Using circle approximations:
- 6-8 um diameter => ~28-50 um^2
- 10-20 um diameter => ~79-314 um^2
- 16-27 um diameter => ~201-573 um^2
- 30-40 um diameter => ~707-1257 um^2

Recommended broad defaults (with margin):
- `area_low = 10 um^2`
- `area_high = 1000 um^2`

These defaults include very small cells and large epithelial/hepatocyte-sized cells while still excluding obvious outliers.


## Data-driven selection (best practice)

1) If you have nucleus and cell areas, compute:
`s = sqrt(median(cell_area) / median(nucleus_area))`
Then add a small margin (e.g., +5% to +15%).

2) For area bounds, use robust percentiles (e.g., 1st-99th) and widen by 10-20%.

3) Confirm visually or with transcript assignment metrics: 
- Too high `s` increases cross-cell mixing.
- Too low `s` misses peripheral transcripts.


In [ ]:
import numpy as np

def recommend_scale_factor(cell_area, nucleus_area, margin=0.1):
    """Return a linear scale factor based on median areas, with margin."""
    s = np.sqrt(np.nanmedian(cell_area) / np.nanmedian(nucleus_area))
    return s * (1.0 + margin)

def recommend_area_bounds(cell_area, low_q=0.01, high_q=0.99, margin=0.1):
    """Return area_low/high based on quantiles with symmetric margin."""
    lo = np.nanquantile(cell_area, low_q)
    hi = np.nanquantile(cell_area, high_q)
    return lo * (1.0 - margin), hi * (1.0 + margin)


In [ ]:
# Example usage with toy values (um^2)
cell_area = np.array([120, 140, 160, 200, 260, 300])
nucleus_area = np.array([30, 35, 40, 45, 50, 55])

s = recommend_scale_factor(cell_area, nucleus_area, margin=0.1)
area_low, area_high = recommend_area_bounds(cell_area, low_q=0.05, high_q=0.95, margin=0.1)

print(f"Recommended prediction_scale_factor: {s:.2f}")
print(f"Recommended area_low: {area_low:.1f} um^2")
print(f"Recommended area_high: {area_high:.1f} um^2")


## Quick checklist

- Verify your coordinate units (Segger defaults assume microns).
- Use *linear* sizes to choose `prediction_scale_factor`.
- Use percentile-based bounds for `area_low/high`, then add a margin.
- Re-check after model training by inspecting cell size distributions and mis-assignment patterns.
